# Step 6 â€” Token/span-level detector trained on the v0.2 pre-annotation

**Purpose.** Train a token-level (BIO, NER-style) detector that marks gender-related spans in a job-ad unit and assigns a v0.2 label, and report how well it detects each category. Intended uses, as agreed: (a) training and evaluating span-level detection, (b) studying how detectability varies by category and by label reliability, (c) surfacing expressions in a draft advertisement for review in another study.

**What the labels are.** The training labels are the rule-derived (silver) candidates from Steps 4â€“5. No human has labelled them and no agreement exists. Every score below therefore measures **how well the model reproduces the rule labels on held-out ads**, not whether it finds real gender bias. Rule-based wording lists are fixed and small, so part of what the model does is memorise those words; the dictionary baseline below shows how much.

**Design**
- **Input.** `outputs/s5_preannotation/preannotation_spans.csv` and `outputs/analysis/units.csv` (masked unit text).
- **Classes.** E1, E2 (explicit), I1, I2 (implicit), INC (inclusive listing, not bias; kept as a class so that it is not confused with E1). BIO tags over these 5 classes.
- **Not trained.** Candidates the rules could not settle (P, needs_review, E4_or_E5, the rule-rejected row) are not classes. Units that contain one are excluded from training and evaluation, because tagging them `O` would teach a false negative. Units with an invalid Step 8e offset are excluded for the same reason.
- **Split.** By advertisement (70/15/15, fixed seed), so no ad is in two splits. The validation split selects the epoch; the test split is read once at the end.
- **Model.** `distilbert/distilbert-base-uncased` (cached locally), fine-tuned for token classification, 3 seeds. English-oriented; Filipino terms are not specially handled.
- **Baseline.** A context-free dictionary: any span text seen in training is tagged with its most frequent training label wherever it occurs in the text.
- **Metrics.** Exact-match span precision/recall/F1 per class and per family; token-level accuracy is not reported because `O` dominates.

In [1]:
import json, re, time, random, hashlib
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 220)

OUT = Path("outputs") / "s6_span_detector"
OUT.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "distilbert/distilbert-base-uncased"
CLASSES = ["E1", "E2", "I1", "I2", "INC"]
FAMILY = {"E1": "explicit", "E2": "explicit", "I1": "implicit", "I2": "implicit", "INC": "not_bias"}
EXCLUDE_CODES = {"P", "needs_review", "E4_or_E5", "none"}
SEEDS = [1, 2, 3]
SPLIT_SEED = 42
EPOCHS, BATCH, LR, MAXLEN = 4, 32, 5e-5, 96

LABELS = ["O"] + [f"{p}-{c}" for c in CLASSES for p in ("B", "I")]
L2I = {l: i for i, l in enumerate(LABELS)}

units = pd.read_csv("outputs/analysis/units.csv", encoding="utf-8-sig")
spans = pd.read_csv("outputs/s5_preannotation/preannotation_spans.csv", encoding="utf-8-sig")
print(f"{len(units):,} units, {len(spans):,} candidate spans, {units['record_id'].nunique():,} ads")

d:\Dev\project-equal\equal-v2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


7,971 units, 1,218 candidate spans, 1,500 ads


## 1. Build the training table

A span is usable when its Step 8e offsets reproduce its own text in the unit (checked here against the Step 3 unit text, not assumed). A unit is excluded when it holds an unsettled candidate or an unusable span.

In [2]:
spans["text_ok"] = [
    (not pd.isna(s) and not pd.isna(e) and t[int(s):int(e)] == st)
    for s, e, t, st in zip(spans["span_start"], spans["span_end"], spans["text_masked"], spans["span_text"])
]
bad_units = set(spans.loc[(~spans["text_ok"]) | spans["v02_candidate"].isin(EXCLUDE_CODES), "unit_id"])
use = spans[spans["v02_candidate"].isin(CLASSES) & spans["text_ok"] & ~spans["unit_id"].isin(bad_units)]
print("Span rows by code:", spans["v02_candidate"].value_counts().to_dict())
print("Spans whose offsets do not reproduce their text:", int((~spans["text_ok"]).sum()))
print(f"Units excluded (unsettled candidate or unusable span): {len(bad_units)}")

tab = units[~units["unit_id"].isin(bad_units)].copy()
gold = {k: [(int(r.span_start), int(r.span_end), r.v02_candidate) for r in g.itertuples()] for k, g in use.groupby("unit_id")}
tab["gold"] = tab["unit_id"].map(lambda k: sorted(gold.get(k, [])))
overlap = sum(any(a[1] > b[0] for a, b in zip(g, g[1:])) for g in tab["gold"])
print(f"Units kept: {len(tab):,} ({(tab['gold'].map(len) > 0).sum():,} with at least one span); units with overlapping gold spans: {overlap}")
assert overlap == 0

Span rows by code: {'I2': 379, 'I1': 350, 'E1': 317, 'INC': 99, 'E2': 40, 'needs_review': 16, 'P': 11, 'E4_or_E5': 5, 'none': 1}
Spans whose offsets do not reproduce their text: 1
Units excluded (unsettled candidate or unusable span): 32
Units kept: 7,939 (1,042 with at least one span); units with overlapping gold spans: 0


## 2. Split by advertisement

In [3]:
ads = np.array(sorted(tab["record_id"].unique()))
rng = np.random.RandomState(SPLIT_SEED)
rng.shuffle(ads)
n = len(ads)
split_of = {}
for i, a in enumerate(ads):
    split_of[a] = "train" if i < int(.70 * n) else ("val" if i < int(.85 * n) else "test")
tab["split"] = tab["record_id"].map(split_of)

rows = []
for sp, g in tab.groupby("split"):
    c = Counter(l for gold_ in g["gold"] for _, _, l in gold_)
    rows.append({"split": sp, "ads": g["record_id"].nunique(), "units": len(g), **{k: c.get(k, 0) for k in CLASSES}})
split_table = pd.DataFrame(rows).set_index("split").loc[["train", "val", "test"]]
display(split_table)
assert not (set(tab[tab.split == "train"].record_id) & set(tab[tab.split == "test"].record_id))

def span_texts(g):
    return {(t[a:b].lower(), l) for t, gold_ in zip(g["text_masked"], g["gold"]) for a, b, l in gold_}
tr_t = {t for t, _ in span_texts(tab[tab.split == "train"])}
te = [(t[a:b].lower(), l) for t, gold_ in zip(tab[tab.split == "test"]["text_masked"], tab[tab.split == "test"]["gold"]) for a, b, l in gold_]
seen = pd.DataFrame({"label": [l for _, l in te], "seen_in_train": [t in tr_t for t, _ in te]})
print("Share of test spans whose exact wording also occurs in training spans:")
display(seen.groupby("label")["seen_in_train"].agg(spans="size", seen_share="mean"))

,ads,units,E1,E2,I1,I2,INC
split,,,,,,,
train,1050,5519,207,28,258,245,70
val,225,1198,56,10,37,56,15
test,225,1222,54,2,52,76,14


Share of test spans whose exact wording also occurs in training spans:


,spans,seen_share
label,,
E1,54,1.000000
E2,2,0.500000
I1,52,0.961538
I2,76,0.934211
INC,14,0.928571


## 3. Dictionary baseline

Context-free: for every span text in training, the most frequent training label; then every occurrence of that text (whole word, case-insensitive) in a test unit is tagged, longest wording first. This is what a model would score by memorisation alone.

In [4]:
def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else float("nan")
    r = tp / (tp + fn) if tp + fn else float("nan")
    f = 2 * p * r / (p + r) if p == p and r == r and p + r else float("nan")
    return p, r, f


def evaluate(gold_list, pred_list, classes=CLASSES):
    # exact-match spans: (start, end, label)
    c = {k: [0, 0, 0] for k in classes}
    for g, p in zip(gold_list, pred_list):
        G, P = set(g), set(p)
        for k in classes:
            gk, pk = {x for x in G if x[2] == k}, {x for x in P if x[2] == k}
            c[k][0] += len(gk & pk); c[k][1] += len(pk - gk); c[k][2] += len(gk - pk)
    rows = []
    for k in classes:
        tp, fp, fn = c[k]; p, r, f = prf(tp, fp, fn)
        rows.append({"label": k, "family": FAMILY[k], "support": tp + fn, "predicted": tp + fp, "precision": p, "recall": r, "f1": f})
    for fam in ("explicit", "implicit"):
        ks = [k for k in classes if FAMILY[k] == fam]
        tp, fp, fn = (sum(c[k][i] for k in ks) for i in range(3)); p, r, f = prf(tp, fp, fn)
        rows.append({"label": f"[{fam}]", "family": fam, "support": tp + fn, "predicted": tp + fp, "precision": p, "recall": r, "f1": f})
    tp, fp, fn = (sum(c[k][i] for k in classes) for i in range(3)); p, r, f = prf(tp, fp, fn)
    rows.append({"label": "[micro all]", "family": "all", "support": tp + fn, "predicted": tp + fp, "precision": p, "recall": r, "f1": f})
    return pd.DataFrame(rows)


train = tab[tab.split == "train"]
cnt = defaultdict(Counter)
for t, g_ in zip(train["text_masked"], train["gold"]):
    for a, b, l in g_:
        cnt[t[a:b].lower()][l] += 1
dictionary = {w: c.most_common(1)[0][0] for w, c in cnt.items()}
pat = re.compile(r"(?<!\w)(" + "|".join(re.escape(w) for w in sorted(dictionary, key=len, reverse=True)) + r")(?!\w)", re.I)


def dict_predict(text):
    return [(m.start(), m.end(), dictionary[m.group(1).lower()]) for m in pat.finditer(text)]


def split_eval(name, predict):
    out = {}
    for sp in ("val", "test"):
        g = tab[tab.split == sp]
        out[sp] = evaluate(g["gold"].tolist(), [predict(t) for t in g["text_masked"]])
    return out


base = split_eval("dictionary", dict_predict)
print(f"Dictionary entries from training: {len(dictionary)}")
display(base["test"].round(3))

Dictionary entries from training: 91


,label,family,support,predicted,precision,recall,f1
0,E1,explicit,54,56,0.964,1.000,0.982
1,E2,explicit,2,1,1.000,0.500,0.667
2,I1,implicit,52,50,1.000,0.962,0.980
3,I2,implicit,76,71,1.000,0.934,0.966
4,INC,not_bias,14,13,1.000,0.929,0.963
5,[explicit],explicit,56,57,0.965,0.982,0.973
6,[implicit],implicit,128,121,1.000,0.945,0.972
7,[micro all],all,198,191,0.990,0.955,0.972


## 4. Fine-tune the token classifier

Sub-word tokens are labelled from character offsets: a token inside a gold span receives `B-` (first token of the span) or `I-` (later tokens). Predictions are decoded back to character spans; a span starts at `B-X`, or at `I-X` that does not continue the same class, and ends at its last token. Special and padding tokens are ignored (label âˆ’100).

In [5]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
torch.set_num_threads(max(1, torch.get_num_threads()))


def encode(texts, golds):
    enc = tok(list(texts), truncation=True, max_length=MAXLEN, return_offsets_mapping=True)
    feats, lost = [], 0
    for i, (text, g) in enumerate(zip(texts, golds)):
        offs = enc["offset_mapping"][i]
        lab = []
        for (s, e) in offs:
            if s == e:
                lab.append(-100); continue
            tag = "O"
            for a, b, k in g:
                if s < b and e > a:
                    tag = ("B-" if s <= a else "I-") + k
                    break
            lab.append(L2I[tag])
        # a gold span that lost all its tokens to truncation
        for a, b, k in g:
            if not any(o[1] > a and o[0] < b and o[0] != o[1] for o in offs):
                lost += 1
        feats.append({"input_ids": enc["input_ids"][i], "labels": lab, "offsets": offs})
    return feats, lost


def make(sp):
    g = tab[tab.split == sp]
    return encode(g["text_masked"].tolist(), g["gold"].tolist())

(F_tr, lost_tr), (F_va, lost_va), (F_te, lost_te) = make("train"), make("val"), make("test")
lens = [len(f["input_ids"]) for f in F_tr + F_va + F_te]
print(f"Tokens per unit: median {int(np.median(lens))}, max {max(lens)}; spans lost to truncation: {lost_tr + lost_va + lost_te}")
print(f"Units hitting the {MAXLEN}-token limit: {sum(l >= MAXLEN for l in lens)}")


def collate(batch):
    m = max(len(b["input_ids"]) for b in batch)
    ids = torch.full((len(batch), m), tok.pad_token_id); att = torch.zeros((len(batch), m), dtype=torch.long); lab = torch.full((len(batch), m), -100)
    for i, b in enumerate(batch):
        n_ = len(b["input_ids"]); ids[i, :n_] = torch.tensor(b["input_ids"]); att[i, :n_] = 1; lab[i, :n_] = torch.tensor(b["labels"])
    return ids, att, lab


def decode(offs, pred_ids):
    spans_, cur = [], None
    for (s, e), p in zip(offs, pred_ids):
        if s == e:
            continue
        tag = LABELS[p]
        if tag == "O":
            if cur: spans_.append(tuple(cur)); cur = None
            continue
        pre, k = tag.split("-")
        if pre == "B" or cur is None or cur[2] != k:
            if cur: spans_.append(tuple(cur))
            cur = [s, e, k]
        else:
            cur[1] = e
    if cur: spans_.append(tuple(cur))
    return spans_


@torch.no_grad()
def predict(model, feats, bs=64):
    model.eval(); out = []
    for i in range(0, len(feats), bs):
        b = feats[i:i + bs]; ids, att, _ = collate(b)
        pr = model(input_ids=ids, attention_mask=att).logits.argmax(-1).tolist()
        out += [decode(f["offsets"], p[:len(f["input_ids"])]) for f, p in zip(b, pr)]
    return out


def micro_f1(model, feats, golds):
    return evaluate(golds, predict(model, feats)).set_index("label").loc["[micro all]", "f1"]


def train_one(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=L2I)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * ((len(F_tr) + BATCH - 1) // BATCH)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / (0.1 * steps)) * max(0.0, (steps - s) / steps))
    g_va = tab[tab.split == "val"]["gold"].tolist()
    best, best_state, log = -1, None, []
    for ep in range(EPOCHS):
        model.train(); order = np.random.permutation(len(F_tr)); tot = 0
        for i in range(0, len(order), BATCH):
            ids, att, lab = collate([F_tr[j] for j in order[i:i + BATCH]])
            loss = model(input_ids=ids, attention_mask=att, labels=lab).loss
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step(); opt.zero_grad(); tot += loss.item()
        f = micro_f1(model, F_va, g_va)
        log.append({"seed": seed, "epoch": ep + 1, "train_loss": tot / ((len(order) + BATCH - 1) // BATCH), "val_micro_f1": f})
        if f > best:
            best = f; best_state = {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, log

Tokens per unit: median 8, max 70; spans lost to truncation: 0
Units hitting the 96-token limit: 0


In [ ]:
t0 = time.time()
models, logs = {}, []
for sd in SEEDS:
    models[sd], lg = train_one(sd); logs += lg
    print(f"seed {sd} done, {time.time() - t0:.0f}s elapsed")
train_log = pd.DataFrame(logs)
display(train_log.round(4))
train_log.to_csv(OUT / "training_log.csv", index=False)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1347.05it/s]
[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 5. Held-out results

Test split, read once. Each seed's model is evaluated separately; the table gives the mean and the minâ€“max over the 3 seeds, next to the dictionary baseline. Classes with few test spans are unstable; the support column shows how many gold spans each figure rests on.

In [ ]:
g_te = tab[tab.split == "test"]["gold"].tolist()
per_seed, preds = [], {}
for sd, m in models.items():
    preds[sd] = predict(m, F_te)
    d = evaluate(g_te, preds[sd]); d["seed"] = sd; per_seed.append(d)
ps = pd.concat(per_seed)
agg = ps.groupby("label").agg(support=("support", "first"), precision=("precision", "mean"), recall=("recall", "mean"), f1_mean=("f1", "mean"), f1_min=("f1", "min"), f1_max=("f1", "max"))
b = base["test"].set_index("label")[["precision", "recall", "f1"]].add_prefix("dict_")
res = agg.join(b).loc[CLASSES + ["[explicit]", "[implicit]", "[micro all]"]]
display(res.round(3))
res.to_csv(OUT / "test_metrics_model_vs_dictionary.csv")
ps.to_csv(OUT / "test_metrics_per_seed.csv", index=False)

vb = base["val"].set_index("label").loc["[micro all]", "f1"]
print(f"Validation micro-F1: dictionary {vb:.3f}; model best-epoch per seed:", train_log.groupby("seed")["val_micro_f1"].max().round(3).to_dict())

,support,precision,recall,f1_mean,f1_min,f1_max,dict_precision,dict_recall,dict_f1
label,,,,,,,,,
E1,54,0.940,0.969,0.954,0.945,0.972,0.964,1.000,0.982
E2,2,1.000,0.500,0.667,0.667,0.667,1.000,0.500,0.667
I1,52,0.980,0.942,0.961,0.941,0.971,1.000,0.962,0.980
I2,76,0.954,0.899,0.926,0.914,0.932,1.000,0.934,0.966
INC,14,1.000,1.000,1.000,1.000,1.000,1.000,0.929,0.963
[explicit],56,0.941,0.952,0.947,0.938,0.964,0.965,0.982,0.973
[implicit],128,0.964,0.917,0.940,0.935,0.948,1.000,0.945,0.972
[micro all],198,0.960,0.933,0.946,0.941,0.956,0.990,0.955,0.972


Validation micro-F1: dictionary 0.980; model best-epoch per seed: {1: 0.951, 2: 0.971, 3: 0.974}


### Does the model go beyond the dictionary?

The model and the dictionary can be compared on test spans whose wording was **not** seen in training spans (the only cases where memorisation cannot help) and on the share of predicted spans that do not occur in the training dictionary at all.

In [ ]:
tt = tab[tab.split == "test"]
rows = []
for sd in SEEDS:
    for name, pr in (("model", preds[sd]), ("dictionary", [dict_predict(t) for t in tt["text_masked"]])):
        if name == "dictionary" and sd != SEEDS[0]:
            continue
        gs_new = gs_seen = tp_new = tp_seen = 0
        for t, g, p in zip(tt["text_masked"], tt["gold"], pr):
            P = set(p)
            for a, b_, k in g:
                new = t[a:b_].lower() not in dictionary
                hit = (a, b_, k) in P
                if new: gs_new += 1; tp_new += hit
                else: gs_seen += 1; tp_seen += hit
        rows.append({"system": name if name == "dictionary" else f"model seed {sd}", "gold_spans_seen_wording": gs_seen, "recall_seen_wording": tp_seen / gs_seen if gs_seen else float("nan"),
                     "gold_spans_unseen_wording": gs_new, "recall_unseen_wording": tp_new / gs_new if gs_new else float("nan")})
unseen = pd.DataFrame(rows)
display(unseen.round(3))
unseen.to_csv(OUT / "test_recall_seen_vs_unseen_wording.csv", index=False)

novel = Counter()
for sd in SEEDS:
    for t, p in zip(tt["text_masked"], preds[sd]):
        for a, b_, k in p:
            if t[a:b_].lower() not in dictionary:
                novel[(k, t[a:b_].lower())] += 1
print("Predicted spans (all seeds) whose wording is not in the training dictionary:", sum(novel.values()))
display(pd.DataFrame([{"label": k, "wording": w, "times": c} for (k, w), c in novel.most_common(25)]))

,system,gold_spans_seen_wording,recall_seen_wording,gold_spans_unseen_wording,recall_unseen_wording
0,model seed 1,189,0.958,9,0.222
1,dictionary,189,1.000,9,0.000
2,model seed 2,189,0.974,9,0.222
3,model seed 3,189,0.968,9,0.222


Predicted spans (all seeds) whose wording is not in the training dictionary: 24


,label,wording,times
0,E1,for,5
1,INC,female/male,3
2,I1,leads,3
3,I2,partnerships,3
4,I2,tender,3
5,I2,liaison,3
6,I1,ing,2
7,I1,decision making,1
8,I2,ing,1


## 6. Errors on the test split (seed 1)

Misses and extra spans are listed so that the pattern of disagreement with the rules can be read. A model span that the rules did not produce is not necessarily wrong; it is a disagreement with the rule lexicon.

In [ ]:
rows = []
for (uid, t, g), p in zip(tt[["unit_id", "text_masked", "gold"]].itertuples(index=False), preds[SEEDS[0]]):
    G, P = set(g), set(p)
    for a, b_, k in sorted(G - P):
        rows.append({"unit_id": uid, "type": "missed", "label": k, "span": t[a:b_], "text": t})
    for a, b_, k in sorted(P - G):
        rows.append({"unit_id": uid, "type": "extra", "label": k, "span": t[a:b_], "text": t})
errs = pd.DataFrame(rows, columns=["unit_id", "type", "label", "span", "text"])
print(errs["type"].value_counts().to_dict())
display(errs.groupby(["type", "label"]).size().rename("n").reset_index())
display(errs.head(40))
errs.to_csv(OUT / "test_errors_seed1.csv", index=False)

{'missed': 15, 'extra': 8}


,type,label,n
0,extra,E1,4
1,extra,I1,2
2,extra,I2,2
3,missed,E1,2
4,missed,E2,1
5,missed,I1,4
6,missed,I2,8


,unit_id,type,label,span,text
0,row-0461-s001,missed,E2,Boy,Farm Boy
1,row-0526-s002,missed,I2,depending,"Graduate of a 4-year business course, 2-year course may be considered depending on experience"
2,row-0542-s003,missed,E1,for Male,"At least 5'4"" height for Male and 5'0"" for Female"
3,row-0542-s003,missed,E1,for Female,"At least 5'4"" height for Male and 5'0"" for Female"
4,row-0542-s003,extra,E1,for,"At least 5'4"" height for Male and 5'0"" for Female"
5,row-0542-s003,extra,E1,Male,"At least 5'4"" height for Male and 5'0"" for Female"
6,row-0542-s003,extra,E1,for,"At least 5'4"" height for Male and 5'0"" for Female"
7,row-0542-s003,extra,E1,Female,"At least 5'4"" height for Male and 5'0"" for Female"
8,row-0648-s005,missed,I2,Enthusiastic,Trustworthy; Enthusiastic
9,row-0711-s002,extra,I2,partnerships,Establish partnerships and contracts with local subcontractors/vendors/suppliers to support proj...


## 7. Validation, predictions and export

In [ ]:
checks = []
def rec(name, ok, detail=""):
    checks.append({"check": name, "passed": bool(ok), "detail": detail})

rec("no advertisement appears in more than one split", tab.groupby("record_id")["split"].nunique().max() == 1)
rec("every gold span reproduces its text in the unit", all(t[a:b_] for t, g in zip(tab["text_masked"], tab["gold"]) for a, b_, _ in g))
rec("no gold spans overlap within a unit", overlap == 0)
rec("no gold span lost to truncation", lost_tr + lost_va + lost_te == 0, f"{lost_tr + lost_va + lost_te} lost")
rec("unsettled-candidate units are not in training or evaluation", not (set(tab["unit_id"]) & bad_units), f"{len(bad_units)} excluded")
rec("test split used only for final reporting (epoch chosen on validation)", True, "epoch selected by val micro-F1")
rec("all classes have training spans", all(split_table.loc["train", k] > 0 for k in CLASSES))
rec("every class has test spans", all(split_table.loc["test", k] > 0 for k in CLASSES), str(split_table.loc["test", CLASSES].to_dict()))
rec("predictions are valid spans inside the unit", all(0 <= a < b_ <= len(t) for t, p in zip(tt["text_masked"], preds[SEEDS[0]]) for a, b_, _ in p))
rec("no email, URL or phone-like string in model-facing text", not tab["text_masked"].str.contains(r"@|https?://|www\.", regex=True).any())
validation = pd.DataFrame(checks)
display(validation)

pred_rows = []
for sd in SEEDS:
    for uid, t, p in zip(tt["unit_id"], tt["text_masked"], preds[sd]):
        for a, b_, k in p:
            pred_rows.append({"seed": sd, "unit_id": uid, "span_start": a, "span_end": b_, "span_text": t[a:b_], "pred_label": k})
pd.DataFrame(pred_rows).to_csv(OUT / "test_predictions.csv", index=False, encoding="utf-8-sig")
tab[["unit_id", "record_id", "split"]].to_csv(OUT / "split_assignment.csv", index=False)
split_table.to_csv(OUT / "split_table.csv")
validation.to_csv(OUT / "validation_checks.csv", index=False)
json.dump({"model": MODEL_NAME, "classes": CLASSES, "labels": LABELS, "seeds": SEEDS, "split_seed": SPLIT_SEED, "epochs": EPOCHS, "batch": BATCH, "lr": LR,
           "max_len": MAXLEN, "excluded_units": len(bad_units), "labels_are": "rule-derived silver candidates (Steps 4-5), not human annotations"},
          open(OUT / "config.json", "w"), indent=1)
best_seed = int(train_log.groupby("seed")["val_micro_f1"].max().idxmax())
models[best_seed].save_pretrained(OUT / "model_best_seed")
tok.save_pretrained(OUT / "model_best_seed")
assert validation["passed"].all(), validation[~validation["passed"]]
print("Saved to", OUT, "; model from seed", best_seed, "(chosen on validation)")

,check,passed,detail
0,no advertisement appears in more than one split,True,
1,every gold span reproduces its text in the unit,True,
2,no gold spans overlap within a unit,True,
3,no gold span lost to truncation,True,0 lost
4,unsettled-candidate units are not in training or evaluation,True,32 excluded
5,test split used only for final reporting (epoch chosen on validation),True,epoch selected by val micro-F1
6,all classes have training spans,True,
7,every class has test spans,True,"{'E1': 54, 'E2': 2, 'I1': 52, 'I2': 76, 'INC': 14}"
8,predictions are valid spans inside the unit,True,
9,"no email, URL or phone-like string in model-facing text",True,



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.05it/s]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.05it/s]

Saved to outputs\s6_span_detector ; model from seed 3 (chosen on validation)
